# Stage 7 — Event-limited full-day intervention validation

Interventions are applied as discrete operational events with cooldown and lead
time. They are not repeated at every five-minute row.

In [ ]:
#@title Shared MFAR paths and stage initialization
from pathlib import Path
from datetime import datetime, timezone
import os, sys, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

# Locate the code repository only; all simulation I/O paths are resolved by
# src.mfar_paths through MFAR_GDRIVE_ROOT or the mounted/synchronized Drive.
_code_candidates = [Path.cwd(), Path.cwd().parent]
if os.environ.get("MFAR_CODE_ROOT"):
    _code_candidates.insert(0, Path(os.environ["MFAR_CODE_ROOT"]))
_code_candidates.extend([
    Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline"),
    Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline/MFAR_Modular_Colab_Pipeline"),
])
for _candidate in _code_candidates:
    if (_candidate / "src" / "mfar_paths.py").is_file():
        sys.path.insert(0, str(_candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "Modul src/mfar_paths.py tidak ditemukan. Jalankan notebook dari repository "
        "atau tetapkan MFAR_CODE_ROOT ke folder repository."
    )

from src.mfar_paths import (
    AIS_RAW_PATH, VEHICLE_ARRIVAL_PATH, DATA_RAW_DIR, CONFIG_DIR,
    STAGE_OUTPUT_DIR, STAGE_01_DIR, STAGE_02_DIR, STAGE_03_DIR,
    STAGE_04_DIR, STAGE_05_DIR, STAGE_06_DIR, STAGE_07_DIR,
    validate_csv_input, validate_raw_inputs, validate_writable_directory,
    write_execution_metadata,
)

_MFAR_STARTED_AT = datetime.now(timezone.utc)

NOTEBOOK_NAME = "07_Candidate_Action.ipynb"
RAW, CFG, STAGE = DATA_RAW_DIR, CONFIG_DIR, STAGE_OUTPUT_DIR
validate_writable_directory(STAGE_07_DIR, NOTEBOOK_NAME, 7)
print("Input stages:", STAGE_04_DIR, STAGE_06_DIR)
print("Output folder:", STAGE_07_DIR)


In [ ]:
queue=validate_csv_input(STAGE_04_DIR/"04_daily_port_queue_forecast.csv", ["simulation_time","port_id","queue_ce","queue_ratio"], NOTEBOOK_NAME, 7)
queue["simulation_time"]=pd.to_datetime(queue["simulation_time"],errors="coerce")
events=validate_csv_input(STAGE_04_DIR/"04_daily_event_log.csv", ["simulation_time","origin","served_ce"], NOTEBOOK_NAME, 7)
events["simulation_time"]=pd.to_datetime(events["simulation_time"],errors="coerce")
rules=validate_csv_input(STAGE_06_DIR/"06_rule_evaluation.csv", ["simulation_time","origin","selected_action","selected_rule_strength","dominant_rule"], NOTEBOOK_NAME, 7)
rules["simulation_time"]=pd.to_datetime(rules["simulation_time"],errors="coerce")
profiles=pd.read_csv(CFG/"vessel_profiles.csv")
capacity=float(profiles["vehicle_capacity_ce"].median())

In [ ]:
# Convert fuzzy recommendations into no more than one intervention episode
# per port-action during the cooldown window.
recommend=rules.rename(columns={"origin":"port_id"}).sort_values("simulation_time")
recommend=recommend[
    recommend["selected_rule_strength"]>=0.35
].copy()

cooldown_min={
 "DEPART_NOW":30,"INCREASE_SERVICE_PRIORITY":45,
 "RESCHEDULE_HEADWAY":60,"ADD_VESSEL":180,
 "HOLD_DEPARTURE":30,"REDUCE_SPEED":30,"ALERT_OPERATOR":60,
 "MAINTAIN_SPEED":30,"NO_INTERVENTION":30
}
accepted=[]
last={}
for _,r in recommend.iterrows():
    key=(str(r["port_id"]),r["selected_action"])
    prev=last.get(key)
    cd=cooldown_min.get(r["selected_action"],60)
    if prev is not None and (r["simulation_time"]-prev).total_seconds()/60<cd:
        continue
    accepted.append(r)
    last[key]=r["simulation_time"]
accepted=pd.DataFrame(accepted)

# Only queue-serving interventions create a discrete service event.
extra=[]
for _,r in accepted.iterrows():
    action=r["selected_action"]
    strength=float(r["selected_rule_strength"])
    if action=="DEPART_NOW":
        lead=10; cap=capacity*min(1.0,strength)
    elif action=="INCREASE_SERVICE_PRIORITY":
        lead=20; cap=capacity*0.5*strength
    elif action=="ADD_VESSEL":
        lead=60; cap=capacity*strength   # one added trip after mobilization
    else:
        continue
    extra.append({
      "simulation_time":r["simulation_time"]+pd.Timedelta(minutes=lead),
      "port_id":str(r["port_id"]),
      "action":action,"served_capacity_ce":cap,
      "rule_strength":strength,"dominant_rule":r["dominant_rule"]
    })
extra=pd.DataFrame(extra)

In [ ]:
# Re-simulate from baseline arrivals inferred from queue increments and actual
# departure services, then add accepted intervention events.
sim=queue.copy().sort_values(["port_id","simulation_time"])
sim["actual_service_ce"]=0.0
for _,e in events.iterrows():
    m=(sim["port_id"].eq(str(e["origin"]))) & sim["simulation_time"].eq(e["simulation_time"])
    sim.loc[m,"actual_service_ce"]+=float(e["served_ce"])

# Recover arrivals: q_t = q_(t-1) + arrivals - actual service
sim["prev_queue"]=sim.groupby("port_id")["queue_ce"].shift().fillna(0)
sim["estimated_arrival_ce"]=(
    sim["queue_ce"]-sim["prev_queue"]+sim["actual_service_ce"]
).clip(lower=0)

sim["intervention_service_ce"]=0.0
if not extra.empty:
    for _,e in extra.iterrows():
        # snap to next 5-minute grid
        t=e["simulation_time"].ceil("5min")
        m=(sim["port_id"].eq(e["port_id"])) & sim["simulation_time"].eq(t)
        sim.loc[m,"intervention_service_ce"]+=float(e["served_capacity_ce"])

after=[]
for port,g in sim.groupby("port_id"):
    q=0.0
    for idx,r in g.sort_values("simulation_time").iterrows():
        q+=float(r["estimated_arrival_ce"])
        q=max(0.0,q-float(r["actual_service_ce"]))
        q=max(0.0,q-float(r["intervention_service_ce"]))
        after.append((idx,q))
for idx,q in after:
    sim.loc[idx,"queue_ce_after"]=q

sim["queue_ratio_after"]=sim["queue_ce_after"]/capacity
sim["critical_baseline"]=sim["queue_ratio"]>=3
sim["critical_after"]=sim["queue_ratio_after"]>=3
sim["critical_case_status"]=np.select(
 [sim["critical_baseline"]&~sim["critical_after"],
  sim["critical_baseline"]&sim["critical_after"],
  ~sim["critical_baseline"]&sim["critical_after"]],
 ["RESOLVED","REMAINING","NEW"],default="SAFE"
)

In [ ]:
daily=[]
for p,g in sim.groupby("port_id"):
    daily.append({
     "port_id":p,
     "max_queue_baseline_ce":float(g["queue_ce"].max()),
     "max_queue_after_ce":float(g["queue_ce_after"].max()),
     "mean_queue_baseline_ce":float(g["queue_ce"].mean()),
     "mean_queue_after_ce":float(g["queue_ce_after"].mean()),
     "critical_duration_baseline_min":int(g["critical_baseline"].sum()*5),
     "critical_duration_after_min":int(g["critical_after"].sum()*5),
     "critical_resolved_rows":int((g["critical_case_status"]=="RESOLVED").sum()),
     "intervention_event_count":int((g["intervention_service_ce"]>0).sum()),
     "total_intervention_service_ce":float(g["intervention_service_ce"].sum())
    })
daily=pd.DataFrame(daily)

overall=pd.DataFrame({
 "metric":["total_critical_baseline","total_critical_after",
 "critical_resolved","critical_remaining","critical_new",
 "accepted_intervention_recommendations","queue_area_reduction_percent"],
 "value":[int(sim["critical_baseline"].sum()),int(sim["critical_after"].sum()),
 int((sim["critical_case_status"]=="RESOLVED").sum()),
 int((sim["critical_case_status"]=="REMAINING").sum()),
 int((sim["critical_case_status"]=="NEW").sum()),len(accepted),
 100*((sim["queue_ce"]*5).sum()-(sim["queue_ce_after"]*5).sum())/
 max((sim["queue_ce"]*5).sum(),1e-9)]
})

S7=STAGE/"stage_07"
sim.to_csv(S7/"07_full_day_baseline_vs_intervention.csv",index=False)
accepted.to_csv(S7/"07_accepted_intervention_events.csv",index=False)
extra.to_csv(S7/"07_queue_service_intervention_events.csv",index=False)
daily.to_csv(S7/"07_daily_validation_by_port.csv",index=False)
overall.to_csv(S7/"07_overall_intervention_validation.csv",index=False)
display(daily); display(overall)

## Keluaran interpretatif otomatis\n\nCSV/JSON dipertahankan untuk kontrak data. Sel berikut membuat keluaran yang dapat dibaca dan dieksplorasi tanpa membuka CSV mentah.

In [ ]:
#@title Export readable Stage 7 outputs
from src.mfar_visuals import stage7_intervention_outputs
_readable_outputs = stage7_intervention_outputs(
    sim, accepted, extra, daily, overall, STAGE_07_DIR
)
print("Readable Stage 7 outputs:")
for _path in _readable_outputs:
    print("-", _path.name)


In [ ]:
#@title Execution metadata and saved-artifact report
_stage_dir = STAGE_07_DIR
_saved_files = sorted(_stage_dir.glob("07_*"))
write_execution_metadata(
    stage=7, notebook=NOTEBOOK_NAME, started_at=_MFAR_STARTED_AT,
    input_paths=[STAGE_04_DIR/'04_daily_port_queue_forecast.csv', STAGE_04_DIR/'04_daily_event_log.csv', STAGE_06_DIR/'06_rule_evaluation.csv', CFG/'vessel_profiles.csv'],
    input_rows={"queue": len(queue)},
    output_rows={"sim": len(sim)},
    output_files=_saved_files,
)
